In [2]:
import sys
import os

!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

sys.path.append('/content/proyecto-transformacion-texto-imagen')

print("✅ Entorno configurado correctamente.")

fatal: destination path 'proyecto-transformacion-texto-imagen' already exists and is not an empty directory.
✅ Entorno configurado correctamente.


In [1]:
import sys, os, gc, json
import numpy as np
import polars as pl
import torch
import joblib
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset
from scipy import stats
from skimage.feature import local_binary_pattern
from skimage.filters import sobel
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
PERMUTATION_SEED = 99   # Seed distinto al del pipeline original (42)
                        # para que la permutación sea reproducible pero no idéntica al split
set_seed(42)            # Seed de PyTorch/HuggingFace igual al original
print("✅ Setup completo")

✅ Setup completo


In [ ]:
PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
ruta = f'{PROJECT_ROOT}/data/processed'

train_df = pl.read_csv(f'{ruta}/train.csv')
val_df   = pl.read_csv(f'{ruta}/validation.csv')
test_df  = pl.read_csv(f'{ruta}/test.csv')

X_train = train_df.get_column('text').to_list()
X_val   = val_df.get_column('text').to_list()
X_test  = test_df.get_column('text').to_list()

# Etiquetas REALES (para val y test — nunca se permutan)
y_train_real = train_df.get_column('manual_classification').to_numpy()
y_val        = val_df.get_column('manual_classification').to_numpy()
y_test       = test_df.get_column('manual_classification').to_numpy()

# Etiquetas PERMUTADAS para train
rng = np.random.default_rng(seed=PERMUTATION_SEED)
y_train_perm = rng.permutation(y_train_real)

# Verificación de sanidad: la distribución de clases debe ser idéntica
# (permutamos orden, no composición)
assert y_train_perm.sum() == y_train_real.sum(), \
    "ERROR: la permutación cambió el balance de clases"

print(f"✓ Train original  — Dep: {y_train_real.sum()} | No-dep: {(y_train_real==0).sum()}")
print(f"✓ Train permutado — Dep: {y_train_perm.sum()} | No-dep: {(y_train_perm==0).sum()}")
print(f"✓ Primeras 10 etiquetas reales:    {y_train_real[:10]}")
print(f"✓ Primeras 10 etiquetas permutadas: {y_train_perm[:10]}")

In [ ]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision_macro': p, 'recall_macro': r, 'f1_macro': f1}

print("✅ Clases auxiliares definidas")

In [ ]:
PERM_CKPT_DIR = f'{PROJECT_ROOT}/models/checkpoints/perm_control'
os.makedirs(PERM_CKPT_DIR, exist_ok=True)

# Hiperparámetros IDÉNTICOS a Exp 4.1
HP = {
    'learning_rate':  1.0643090454382045e-05,
    'batch_size':     8,
    'num_epochs':     4,
    'weight_decay':   0.02804067960331229,
    'warmup_ratio':   0.11034302016226369,
    'max_length':     256
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_perm = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Dataset con etiquetas PERMUTADAS en train, REALES en val
train_ds_perm = DepressionDataset(X_train, y_train_perm, tokenizer, HP['max_length'])
val_ds        = DepressionDataset(X_val,   y_val,         tokenizer, HP['max_length'])

training_args = TrainingArguments(
    output_dir                  = PERM_CKPT_DIR,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_strategy            = 'epoch',
    learning_rate               = HP['learning_rate'],
    per_device_train_batch_size = HP['batch_size'],
    per_device_eval_batch_size  = HP['batch_size'],
    num_train_epochs            = HP['num_epochs'],
    weight_decay                = HP['weight_decay'],
    warmup_ratio                = HP['warmup_ratio'],
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    save_total_limit            = 1,
    fp16                        = True,
    seed                        = 42,
    report_to                   = 'none',
)

trainer_perm = Trainer(
    model           = model_perm,
    args            = training_args,
    train_dataset   = train_ds_perm,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("... Entrenando BETO con etiquetas permutadas...")
print(f"   Hiperparámetros: {HP}")
trainer_perm.train()

# Evaluar en val con etiquetas reales — esperamos rendimiento ~azar (~0.50 F1-Macro)
metrics_val = trainer_perm.evaluate()
print(f"\n... F1-Val con etiquetas permutadas: {metrics_val['eval_f1_macro']:.4f}")
print(f"   (Esperado: cercano a 0.50 — rendimiento de azar)")

In [ ]:
from transformers import BertForSequenceClassification
import gc

def extraer_embeddings(texts, model_or_path, tokenizer, batch_size=16, max_length=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if isinstance(model_or_path, str):
        model_cls = BertForSequenceClassification.from_pretrained(
            model_or_path, ignore_mismatched_sizes=True).to(device)
    else:
        model_cls = model_or_path.to(device)

    encoder = model_cls.bert
    encoder.eval()
    all_emb = []

    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out = encoder(**inputs)
            all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())

    del model_cls, encoder
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(all_emb)

print("--> Extrayendo embeddings del modelo permutado...")
emb_train_perm = extraer_embeddings(X_train, trainer_perm.model, tokenizer)
emb_val_perm   = extraer_embeddings(X_val,   trainer_perm.model, tokenizer)
emb_test_perm  = extraer_embeddings(X_test,  trainer_perm.model, tokenizer)

print(f". . .Embeddings extraídos — Train: {emb_train_perm.shape} | Val: {emb_val_perm.shape} | Test: {emb_test_perm.shape}")

# Liberar memoria del modelo permutado — ya no lo necesitas
del trainer_perm, model_perm
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Cargar el UMAP y scaler compartidos — fiteados sobre X_train real de Exp 4.1
pkg_umap = joblib.load(f'{PROJECT_ROOT}/data/shared/umap_params.pkl')

# Verificar estructura — ajusta las claves según cómo Santiago guardó el pkl
print("Claves en umap_params.pkl:", list(pkg_umap.keys()))
# Esperado: algo como {'reducer': <UMAP>, 'scaler': <MinMaxScaler>, ...}
# o posiblemente el pkl contiene directamente los componentes ya transformados

reducer = pkg_umap['reducer']  # <-- ajustar nombre de clave según el pkl real
scaler  = pkg_umap['scaler']   # <-- ajustar nombre de clave según el pkl real

# Transformar (NO fit) los embeddings permutados con el UMAP del pipeline original
umap_train_perm = np.clip(scaler.transform(reducer.transform(emb_train_perm)), 0.0, 1.0)
umap_val_perm   = np.clip(scaler.transform(reducer.transform(emb_val_perm)),   0.0, 1.0)
umap_test_perm  = np.clip(scaler.transform(reducer.transform(emb_test_perm)),  0.0, 1.0)

print(f"✅ UMAP aplicado — Train: {umap_train_perm.shape}")
print(f"   Rango train: [{umap_train_perm.min():.4f}, {umap_train_perm.max():.4f}]")
print(f"   Rango test:  [{umap_test_perm.min():.4f}, {umap_test_perm.max():.4f}]")

In [ ]:
# Opción A: calcular features directamente sobre componentes UMAP
# (válido como control de separabilidad en el espacio reducido)

def calcular_features_umap(umap_components, labels):
    """
    Calcula estadísticas básicas de separabilidad sobre los 5 componentes UMAP.
    No requiere generar imágenes — es un control en el espacio de parámetros.
    """
    resultados = {}
    for i in range(umap_components.shape[1]):
        comp = umap_components[:, i]
        dep    = comp[labels == 1]
        nodep  = comp[labels == 0]
        t_stat, p_val = stats.ttest_ind(dep, nodep, equal_var=False)  # Welch
        resultados[f'umap_{i}'] = {
            'mean_dep':   dep.mean(),
            'mean_nodep': nodep.mean(),
            'std_dep':    dep.std(),
            'std_nodep':  nodep.std(),
            't_stat':     t_stat,
            'p_value':    p_val
        }
    return resultados

# Análisis sobre el conjunto completo (train+val+test) con etiquetas REALES
# para evaluar si los embeddings permutados mantienen separabilidad
umap_all_perm   = np.vstack([umap_train_perm, umap_val_perm, umap_test_perm])
y_all_real      = np.concatenate([y_train_real, y_val, y_test])

features_perm = calcular_features_umap(umap_all_perm, y_all_real)

print("\n📊 Separabilidad inter-clase en componentes UMAP — modelo permutado")
print(f"{'Comp':<8} {'Mean Dep':>10} {'Mean NoDep':>10} {'t-stat':>8} {'p-value':>10} {'Sig':>5}")
print("-" * 55)
n_sig = 0
for comp, vals in features_perm.items():
    sig = "✓" if vals['p_value'] < 0.05 else "—"
    if vals['p_value'] < 0.05:
        n_sig += 1
    print(f"{comp:<8} {vals['mean_dep']:>10.4f} {vals['mean_nodep']:>10.4f} "
          f"{vals['t_stat']:>8.3f} {vals['p_value']:>10.4f} {sig:>5}")

print(f"\n✅ Componentes significativos: {n_sig}/5")
print(f"   (Esperado si control es válido: 0 o 1 — sin separabilidad sistemática)")